# 03 — History personalization and MMR diversification (T-011/S5)

Components verified: history profile aggregation (recency-weighted centroid vs multi-interest per type cluster), personalized scoring `s = (1−β)·r(q,x) + β·cos(p,x)` on top of the hybrid relevance `r` (D-030), cold-start readiness, and MMR re-ranking of the top-50 candidates to 10 (`docs/AI_MVP_DESIGN.md` §7, §9).

**Synthetic users (seeded, labelled synthetic).** Each user has 1–3 interests (top-level OFF categories). A 20-event history is drawn from them; the newest 5 events favour a *current mission* interest (80 %). The scanned query product comes from the current mission, except for 20 % *out-of-interest* scans. Graded relevance: **2** = same leaf category as the query (a similar product), **1** = inside the user's interests, **0** otherwise. Parameters are tuned on 500 users and reported once on 1,000 disjoint test users. No real user data is used.

In [1]:
import json, hashlib, platform, time
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import os, sys; os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from ml.asap_ml import retrieval as R, ranking as K
OUT = ROOT / 'notebooks/results/03_personalization_mmr'; OUT.mkdir(parents=True, exist_ok=True)
SEED = 20261006
cat = R.load_catalog(); N = len(cat); E = np.load('data/processed/embeddings_e5-small.npy'); clusters = np.load('data/processed/cluster_labels.npy')
nb01 = json.loads(Path('notebooks/results/01_data_embeddings_retrieval/summary.json').read_text())
nb02 = json.loads(Path('notebooks/results/02_clustering_pca/summary.json').read_text())
assert hashlib.sha256(E.tobytes()).hexdigest() == nb01['embeddings_sha256']
assert hashlib.sha256(Path('data/processed/cluster_labels.npy').read_bytes()).hexdigest() == nb02['artifact_sha256']['cluster_labels.npy']
ALPHA = nb01['hybrid']['alpha']['no_category']  # 0.9, D-030
T = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True).fit_transform([R.product_text(p) for p in cat]).tocsr()
def relevance(q): return ALPHA * (E @ E[q]) + (1 - ALPHA) * (T @ T[q].T).toarray().ravel()
env = {'python': platform.python_version(), 'numpy': np.__version__, 'run_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'alpha': ALPHA}; env

{'python': '3.13.0',
 'numpy': '2.5.3',
 'run_utc': '2026-10-06T14:00:11Z',
 'alpha': 0.9}

## 1. Synthetic users

In [2]:
top = np.array([p['top_category'] for p in cat]); leaf = np.array([p['category'] for p in cat])
by_top = {t: np.flatnonzero(top == t) for t in np.unique(top)}; tops = sorted(t for t, ix in by_top.items() if len(ix) >= 30)
leaf_size = pd.Series(leaf).value_counts()
def make_users(n, seed):
    rng = np.random.default_rng(seed); users = []
    while len(users) < n:
        interests = list(rng.choice(tops, rng.integers(1, 4), replace=False)); current = interests[0]
        hist = [int(rng.choice(by_top[current if (i < 5 and rng.random() < 0.8) else rng.choice(interests)])) for i in range(20)]
        outside = rng.random() < 0.2
        qtop = rng.choice([t for t in tops if t not in interests]) if outside else current
        pool = [i for i in by_top[qtop] if i not in hist and leaf_size[leaf[i]] >= 2]
        if not pool: continue
        q = int(rng.choice(pool)); interest_set = set(interests)
        gains = np.zeros(N, np.int8); gains[np.isin(top, interests)] = 1; gains[leaf == leaf[q]] = 2; gains[q] = 0; gains[hist] = 0
        users.append({'interests': interests, 'hist': hist, 'q': q, 'outside': bool(outside), 'gains': gains})
    return users
tune, test = make_users(500, SEED), make_users(1000, SEED + 1)
pd.Series({'tune_users': len(tune), 'test_users': len(test), 'test_out_of_interest_share': np.mean([u['outside'] for u in test]),
           'mean_interests': np.mean([len(u['interests']) for u in test]), 'eligible_top_categories': len(tops)})

tune_users                     500.000
test_users                    1000.000
test_out_of_interest_share       0.215
mean_interests                   1.984
eligible_top_categories         45.000
dtype: float64

In [3]:
for u in tune + test: u['rel'] = relevance(u['q'])  # cached hybrid relevance per user query
COLLAPSE = True  # S6c.1: one result per (name, brand) variant group; the query's own variants removed
keys = np.array([K.variant_key(p) for p in cat])
def rank(u, method, beta=0.0, H=5.0):
    if method == 'generic' or K.readiness(u['hist']) == 'COLD_START': s = u['rel']
    else:
        p = K.centroid_profile(E, u['hist'], H) if method == 'centroid' else K.multi_interest_profile(E, u['hist'], H, clusters, E[u['q']])
        s = K.personalized_scores(u['rel'], E, p, beta)
    ex = {u['q'], *u['hist']}
    return s, (K.collapsed_candidates(s, ex, keys, keys[u['q']]) if COLLAPSE else K.candidates(s, ex))
def score_users(users, method, beta=0.0, H=5.0):
    return np.array([K.graded_ndcg(rank(u, method, beta, H)[1], u['gains']) for u in users])

## 2. Tuning β and half-life H (tuning users only)

In [4]:
betas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]; Hs = [3.0, 5.0, 10.0, np.inf]
grid = []
for m in ('centroid', 'multi_interest'):
    for b in betas:
        for H in Hs: grid.append({'method': m, 'beta': b, 'H': H, 'ndcg@10': score_users(tune, m, b, H).mean()})
grid = pd.DataFrame(grid); generic_tune = score_users(tune, 'generic').mean()
best = grid.loc[grid.groupby('method')['ndcg@10'].idxmax()].set_index('method')
CHOSEN = best['ndcg@10'].idxmax(); BETA, H = float(best.loc[CHOSEN, 'beta']), float(best.loc[CHOSEN, 'H'])
fig, ax = plt.subplots(figsize=(7, 3.5))
for m in ('centroid', 'multi_interest'):
    g = grid[(grid.method == m) & (grid.H == best.loc[m, 'H'])]; ax.plot(g.beta, g['ndcg@10'], marker='o', label=f"{m} (H={best.loc[m, 'H']})")
ax.axhline(generic_tune, ls='--', c='grey', label='generic'); ax.set_xlabel('β'); ax.set_ylabel('graded nDCG@10'); ax.legend(); ax.set_title('Personalization tuning (500 users)')
plt.tight_layout(); plt.savefig(OUT / 'beta_sweep.png', dpi=150); plt.show()
print('chosen:', CHOSEN, 'beta =', BETA, 'H =', H, '| generic =', round(generic_tune, 4)); best.round(4)

chosen: multi_interest beta = 0.4 H = inf | generic = 0.7287


/tmp/ipykernel_271158/2391238225.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'beta_sweep.png', dpi=150); plt.show()


,beta,H,ndcg@10
method,,,
centroid,0.4,inf,0.7356
multi_interest,0.4,inf,0.7372


## 3. Test: personalized vs generic (1,000 disjoint users)
Paired bootstrap 95 % CI of the nDCG@10 difference; split by in-interest and out-of-interest scans.

In [5]:
res = {m: score_users(test, m, BETA if m != 'generic' else 0.0, H) for m in ('generic', 'centroid', 'multi_interest')}
res['centroid'] = score_users(test, 'centroid', float(best.loc['centroid', 'beta']), float(best.loc['centroid', 'H']))
res['multi_interest'] = score_users(test, 'multi_interest', float(best.loc['multi_interest', 'beta']), float(best.loc['multi_interest', 'H']))
outside = np.array([u['outside'] for u in test]); brng = np.random.default_rng(SEED + 2)
def ci(d):
    bs = [d[brng.integers(0, len(d), len(d))].mean() for _ in range(1000)]
    return [round(float(d.mean()), 4), *np.round(np.percentile(bs, [2.5, 97.5]), 4).tolist()]
def top10(u, m):
    b = 0.0 if m == 'generic' else float(best.loc[m, 'beta']); h = 5.0 if m == 'generic' else float(best.loc[m, 'H'])
    return rank(u, m, b, h)[1][:10]
def stats(m):
    t10 = [top10(u, m) for u in test]; g10 = [top10(u, 'generic') for u in test]
    return {'ndcg@10': round(float(res[m].mean()), 4), 'ndcg@10_in_interest': round(float(res[m][~outside].mean()), 4), 'ndcg@10_out_of_interest': round(float(res[m][outside].mean()), 4),
            'similar_share@10': round(float(np.mean([np.mean([u['gains'][i] == 2 for i in t]) for u, t in zip(test, t10)])), 4),
            'interest_share@10': round(float(np.mean([np.mean([u['gains'][i] >= 1 for i in t]) for u, t in zip(test, t10)])), 4),
            'rank_changed': round(float(np.mean([not np.array_equal(a, b) for a, b in zip(t10, g10)])), 4)}
pers = pd.DataFrame({m: stats(m) for m in res}).T
diffs = {f'{m}_minus_generic': {'all': ci(res[m] - res['generic']), 'out_of_interest': ci((res[m] - res['generic'])[outside])} for m in ('centroid', 'multi_interest')}
display(pers); diffs

,ndcg@10,ndcg@10_in_interest,ndcg@10_out_of_interest,similar_share@10,interest_share@10,rank_changed
generic,0.7178,0.7403,0.6357,0.5531,0.7814,0.000
centroid,0.7197,0.7463,0.6226,0.5441,0.8017,0.994
multi_interest,0.7251,0.7516,0.6280,0.5469,0.8135,0.995


{'centroid_minus_generic': {'all': [0.0018, -0.0026, 0.0064],
  'out_of_interest': [-0.0131, -0.0231, -0.0038]},
 'multi_interest_minus_generic': {'all': [0.0072, 0.0017, 0.0124],
  'out_of_interest': [-0.0076, -0.0175, 0.0029]}}

## 4. Cold start and readiness
With fewer than 3 distinct known products the request is `COLD_START` and the ranking must equal the generic one exactly.

In [6]:
cold = []
for hist in ([], [test[0]['hist'][0]], test[0]['hist'][:2], [test[0]['hist'][0]] * 5):
    u = dict(test[0], hist=hist); s, c = rank(u, CHOSEN, BETA, H); g = rank(dict(u), 'generic')[1]
    cold.append({'history_len': len(hist), 'distinct': len(set(hist)), 'readiness': K.readiness(hist), 'equals_generic': bool(np.array_equal(c, g))})
u3 = dict(test[0], hist=test[0]['hist'][:3]); cold.append({'history_len': 3, 'distinct': len(set(u3['hist'])), 'readiness': K.readiness(u3['hist']),
    'equals_generic': bool(np.array_equal(rank(u3, CHOSEN, BETA, H)[1], rank(u3, 'generic')[1]))})
cold = pd.DataFrame(cold); cold

,history_len,distinct,readiness,equals_generic
0,0,0,COLD_START,True
1,1,1,COLD_START,True
2,2,2,COLD_START,True
3,5,1,COLD_START,True
4,3,3,SUFFICIENT,False


## 5. MMR λ sweep
Relevance–diversity trade-off on the chosen personalized scores. Rule (pre-declared): λ* = the smallest λ whose tuning nDCG@10 ≥ 95 % of λ = 1 (relevance only); report test values once.

In [7]:
lams = np.round(np.arange(0, 1.01, 0.1), 2)
def mmr_eval(users, lam):
    nd, dv, cov, lc = [], [], [], []
    for u in users:
        s, c = rank(u, CHOSEN, BETA, H); top = K.mmr(c, s, E, lam)
        nd.append(K.graded_ndcg(top, u['gains'])); dv.append(K.ild(top, E)); cov.append(len(set(clusters[top]))); lc.append(len(set(leaf[top])))
    return {'ndcg@10': np.mean(nd), 'ild@10': np.mean(dv), 'cluster_coverage@10': np.mean(cov), 'leaf_coverage@10': np.mean(lc)}
msw = pd.DataFrame({l: mmr_eval(tune, l) for l in lams}).T
LAM = float(msw[msw['ndcg@10'] >= 0.95 * msw.loc[1.0, 'ndcg@10']].index.min())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(msw['ild@10'], msw['ndcg@10'], marker='o')
for l, r in msw.iterrows(): ax[0].annotate(f'{l:.1f}', (r['ild@10'], r['ndcg@10']), fontsize=7)
ax[0].set_xlabel('ILD@10 (diversity)'); ax[0].set_ylabel('graded nDCG@10'); ax[0].set_title('MMR trade-off (tuning users), labels = λ')
msw[['cluster_coverage@10', 'leaf_coverage@10']].plot(ax=ax[1], marker='o', title='Coverage of 10 results'); ax[1].axvline(LAM, ls='--', c='grey')
plt.tight_layout(); plt.savefig(OUT / 'mmr_tradeoff.png', dpi=150); plt.show(); print('λ* =', LAM); msw.round(4)

λ* = 0.7


/tmp/ipykernel_271158/3756279509.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'mmr_tradeoff.png', dpi=150); plt.show(); print('λ* =', LAM); msw.round(4)


,ndcg@10,ild@10,cluster_coverage@10,leaf_coverage@10
0.0,0.5487,0.1439,3.476,6.414
0.1,0.5534,0.1438,3.446,6.332
0.2,0.5641,0.1435,3.400,6.260
0.3,0.5813,0.1425,3.288,6.114
0.4,0.6116,0.1401,3.068,5.802
0.5,0.6563,0.1347,2.834,5.372
0.6,0.7000,0.1257,2.540,4.910
0.7,0.7260,0.1177,2.296,4.572
0.8,0.7394,0.1123,2.122,4.316
0.9,0.7409,0.1082,1.992,4.128


In [8]:
mmr_test = pd.DataFrame({'relevance_only (λ=1)': mmr_eval(test, 1.0), f'mmr (λ={LAM})': mmr_eval(test, LAM)}).T; mmr_test.round(4)

,ndcg@10,ild@10,cluster_coverage@10,leaf_coverage@10
relevance_only (λ=1),0.7251,0.1052,1.891,3.844
mmr (λ=0.7),0.7155,0.1191,2.313,4.521


## 5b. Variant collapse (S6c.1)
Same normalized name + brand = package/size variants (4.1 % of the catalog). Compare the final MMR top-10 on test users without and with the collapse: share of lists that contain variants of each other or of the query, and graded nDCG@10.

In [9]:
def variant_stats(collapse):
    global COLLAPSE
    COLLAPSE = collapse; dup, nd = [], []
    for u in test:
        s, c = rank(u, CHOSEN, BETA, H); top = K.mmr(c, s, E, LAM); ks = list(keys[top])
        dup.append(len(set(ks)) < len(ks) or keys[u['q']] in ks); nd.append(K.graded_ndcg(top, u['gains']))
    COLLAPSE = True
    return {'lists_with_variants': float(np.mean(dup)), 'ndcg@10': float(np.mean(nd))}
variants = pd.DataFrame({'without_collapse': variant_stats(False), 'with_collapse': variant_stats(True)}).T; variants.round(4)

,lists_with_variants,ndcg@10
without_collapse,0.052,0.7170
with_collapse,0.000,0.7155


## 6. MMR edge cases

In [10]:
u = test[0]; s, c = rank(u, CHOSEN, BETA, H)
E2 = np.vstack([E, E[c[0]]]); s2 = np.append(s, s[c[0]] - 1e-6); c2 = np.insert(c, 1, len(E))  # exact duplicate of the top item
edge = {'lambda1_equals_relevance_order': bool(np.array_equal(K.mmr(c, s, E, 1.0), c[:10])),
        'short_pool_returns_all': len(K.mmr(c[:4], s, E, LAM)) == 4, 'empty_pool': len(K.mmr(np.array([], int), s, E, LAM)) == 0,
        'duplicate_position_relevance_only': int(np.flatnonzero(K.mmr(c2, s2, E2, 1.0) == len(E))[0]),
        'duplicate_position_mmr': (lambda r: int(np.flatnonzero(r == len(E))[0]) if (r == len(E)).any() else None)(K.mmr(c2, s2, E2, LAM)),
        'no_repeats': len(set(K.mmr(c, s, E, LAM))) == 10}; edge

{'lambda1_equals_relevance_order': True,
 'short_pool_returns_all': True,
 'empty_pool': True,
 'duplicate_position_relevance_only': 1,
 'duplicate_position_mmr': None,
 'no_repeats': True}

## 7. Runtime of one personalized request (desktop CPU, catalog 10k)

In [11]:
u = test[1]
def full_request():
    r = relevance(u['q']); p = K.multi_interest_profile(E, u['hist'], H, clusters, E[u['q']]) if CHOSEN == 'multi_interest' else K.centroid_profile(E, u['hist'], H)
    s = K.personalized_scores(r, E, p, BETA); return K.mmr(K.collapsed_candidates(s, {u['q'], *u['hist']}, keys, keys[u['q']]), s, E, LAM)
rt = {'full_request': R.timed(full_request, 100), 'mmr_only': R.timed(lambda: K.mmr(c, s, E, LAM), 200)}; rt

{'full_request': {'median_ms': 11.244705998251447,
  'p95_ms': 15.237384604188263},
 'mmr_only': {'median_ms': 0.22744350280845538, 'p95_ms': 0.23926899557409342}}

## 8. Export

In [12]:
summary = {'env': env, 'users': {'tune': len(tune), 'test': len(test)}, 'chosen': {'method': CHOSEN, 'beta': BETA, 'half_life': H, 'mmr_lambda': LAM,
  'min_distinct_history': K.MIN_DISTINCT, 'history_window': 20, 'candidate_pool': 50, 'results': 10},
  'tuning_best': {m: {c: (None if not np.isfinite(v) else round(float(v), 4)) for c, v in r.items()} for m, r in best.iterrows()}, 'generic_tune_ndcg@10': round(float(generic_tune), 4),
  'test': pers.to_dict(orient='index'), 'test_diffs_vs_generic': diffs, 'cold_start': cold.to_dict(orient='records'),
  'mmr_sweep_tune': {str(l): {c: round(float(v), 4) for c, v in r.items()} for l, r in msw.iterrows()},
  'mmr_test': {i: {c: round(float(v), 4) for c, v in r.items()} for i, r in mmr_test.iterrows()}, 'mmr_edge_cases': edge, 'runtime_ms': rt,
  'variant_collapse': {i: {c: round(float(v), 4) for c, v in r.items()} for i, r in variants.iterrows()}}
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2, default=lambda o: None if isinstance(o, float) and not np.isfinite(o) else str(o)))
grid.to_csv(OUT / 'personalization_grid.csv', index=False); msw.round(4).to_csv(OUT / 'mmr_sweep.csv'); pers.to_csv(OUT / 'personalization_test.csv')
print(json.dumps({k: summary[k] for k in ('chosen', 'test_diffs_vs_generic')}, indent=1))

{
 "chosen": {
  "method": "multi_interest",
  "beta": 0.4,
  "half_life": Infinity,
  "mmr_lambda": 0.7,
  "min_distinct_history": 3,
  "history_window": 20,
  "candidate_pool": 50,
  "results": 10
 },
 "test_diffs_vs_generic": {
  "centroid_minus_generic": {
   "all": [
    0.0018,
    -0.0026,
    0.0064
   ],
   "out_of_interest": [
    -0.0131,
    -0.0231,
    -0.0038
   ]
  },
  "multi_interest_minus_generic": {
   "all": [
    0.0072,
    0.0017,
    0.0124
   ],
   "out_of_interest": [
    -0.0076,
    -0.0175,
    0.0029
   ]
  }
 }
}


## Limitations
- Users, interests and relevance are synthetic and derived from OFF categories; the experiment shows the mechanism works as designed and how to set β/H/λ, not real-user satisfaction (that needs the user study).
- Interest membership uses top-level categories while multi-interest grouping uses type clusters; they are correlated, so the multi-interest advantage may be optimistic.
- Timings are desktop CPU; phone/network latency is measured in S8.